In [ ]:
import similaritymeasures
import numpy as np
from scipy.integrate import simpson
from scipy.spatial.distance import euclidean
from scipy.interpolate import interp1d
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['text.usetex'] = True  # Si tienes LaTeX instalado
mpl.rcParams['font.family'] = 'serif'

In [ ]:
#--------------------------------- Estandarización
def estand(f):
    return (f-np.mean(f))/np.std(f)
#--------------------------------- DISTANCIAS L1 Y L2

def distancia_L2(f, g, x):
    return np.sqrt(simpson((f - g)**2, x))

def distancia_L1(f, g, x):
    return simpson(np.abs(f - g), x)

#----------------------------- CORRELACIÓN DE PEARSON FUNCIONAL

def pearson_funcional(f, g, x):
    f_mean = simpson(f, x) / (x[len(x)-1] - x[0])
    g_mean = simpson(g, x) / (x[len(x)-1] - x[0])
    numerator = simpson((f - f_mean) * (g - g_mean), x)
    denominator = np.sqrt(simpson((f - f_mean)**2, x) * simpson((g - g_mean)**2, x))
    return numerator / denominator

#----------------------------- Error Relativo Promedio
def ErrorRelativoPromedio(f,g,x):
    return np.abs(np.sum((f-g)/f)/len(x))

#---------------------------- Metrica
def ErrMetric(data):
    return pearson_funcional(estand(data["SlowLight"]), estand(data["FastLight"]), estand(data["Time"]))

In [ ]:
path_i17_s094 = rf
path_i30_s094 = rf
path_i60_s094 = rf
path_i17_s05  = rf
path_i30_s05  = rf
path_i60_s05  = rf



data_i17_s094 = pd.read_csv(path_i17_s094)
data_i30_s094 = pd.read_csv(path_i30_s094)
data_i60_s094 = pd.read_csv(path_i60_s094)
data_i17_s05  = pd.read_csv(path_i17_s05)
data_i30_s05  = pd.read_csv(path_i30_s05)
data_i60_s05  = pd.read_csv(path_i60_s05)

Err_i17_s094 = ErrMetric(data_i17_s094)
Err_i30_s094 = ErrMetric(data_i30_s094)
Err_i60_s094 = ErrMetric(data_i60_s094)
Err_i17_s05  = ErrMetric(data_i17_s05 )
Err_i30_s05  = ErrMetric(data_i30_s05 )
Err_i60_s05  = ErrMetric(data_i60_s05 )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
import re


datos = {
    'Err_i17_s0.94': r_i17_s094, 
    'Err_i30_s0.94': r_i30_s094, 
    'Err_i60_s0.94': r_i60_s094, 
    'Err_i17_s0.5':  r_i17_s05 , 
    'Err_i30_s0.5':  r_i30_s05 , 
    'Err_i60_s0.5':  r_i60_s05 , 
}


i_values = []
s_values = []
z_values = []

for nombre, valor in datos.items():
    # Usar expresión regular para extraer los números
    match = re.search(r'Err_i(\d+)_s(\d+)', nombre)
    if match:
        i = int(match.group(1))
        s = float(match.group(2)) 
        
        i_values.append(i)
        s_values.append(s)
        z_values.append(valor)

i_array = np.array(i_values)
s_array = np.array(s_values)
z_array = np.array(z_values)

print("Puntos de datos:")
for i, s, z in zip(i_array, s_array, z_array):
    print(f"i={i}, s={s:.2f}, Error={z}")

# Crear una malla más densa para interpolación
i_min, i_max = i_array.min(), i_array.max()
s_min, s_max = s_array.min(), s_array.max()

# Malla densa para interpolación
i_dense = np.linspace(i_min, i_max, 100)
s_dense = np.linspace(s_min, s_max, 100)
I_dense, S_dense = np.meshgrid(i_dense, s_dense)

# Interpolación usando scipy
# Métodos disponibles: 'linear', 'cubic', 'nearest'
Z_interp = griddata((i_array, s_array), z_array, (I_dense, S_dense), 
                    method='cubic', fill_value=np.nan)

# Crear la figura
plt.figure(figsize=(12, 8))

# Mapa de color
im = plt.contourf(I_dense, S_dense, Z_interp, levels=50, cmap='viridis')
plt.colorbar(im, label='Error Value')

# Puntos de datos originales
plt.scatter(i_array, s_array, c=z_array, edgecolors='white', 
           s=100, cmap='viridis', linewidth=2, label='Datos originales')

# Etiquetas y título
plt.xlabel('Ángulo de Inclinación (i) [grados]', fontsize=12)
plt.ylabel('Factor de Spin (s)', fontsize=12)
plt.title('Mapa de Color de Error - Interpolación Cúbica', fontsize=14)
plt.legend()

# Añadir etiquetas de valores en los puntos
for i, (x, y, z) in enumerate(zip(i_array, s_array, z_array)):
    plt.annotate(f'{z:.3f}', (x, y), xytext=(5, 5), 
                textcoords='offset points', fontsize=8, 
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

plt.tight_layout()
plt.show()

# Versión alternativa con pcolormesh (menos suave pero más rápida)
plt.figure(figsize=(12, 8))

# Interpolación lineal para pcolormesh
Z_interp_linear = griddata((i_array, s_array), z_array, (I_dense, S_dense), 
                          method='linear', fill_value=np.nan)

im2 = plt.pcolormesh(I_dense, S_dense, Z_interp_linear, shading='auto', cmap='plasma')
plt.colorbar(im2, label='Error Value')

plt.scatter(i_array, s_array, c='red', s=50, edgecolors='white', label='Datos originales')
plt.xlabel('Ángulo de Inclinación (i) [grados]', fontsize=12)
plt.ylabel('Factor de Spin (s)', fontsize=12)
plt.title('Mapa de Color de Error - pcolormesh', fontsize=14)
plt.legend()

plt.tight_layout()
plt.show()

# Información sobre los datos
print(f"\nResumen de datos:")
print(f"Rango de i: {i_min} - {i_max} grados")
print(f"Rango de s: {s_min:.2f} - {s_max:.2f}")
print(f"Rango de errores: {z_array.min():.3f} - {z_array.max():.3f}")